In [ ]:
# NOTE : This extracts DASCH plates, for testing

In [ ]:
# Installed Python extension via VS Code extensions, and also manually, and also visual studio tools

In [ ]:
# Standalone cell: V CrA -- download ONLY the plates DASCH does NOT reject
# ("accepted" = star detected on the plate AND not flagged by
#  lc.apply_standard_rejections(), i.e. Stage 4 survivors from the funnel)

from IPython.display import HTML, display, clear_output
from daschlab import open_session
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import threading
import time
import pandas as pd
import ipywidgets as widgets

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout {
    color: white !important;
}
.dark-log-output,
.dark-log-output .jp-OutputArea-output,
.dark-log-output .jp-OutputArea-child,
.dark-log-output .output_area,
.dark-log-output pre {
    background-color: #111 !important;
    color: white !important;
    border: none !important;
}
</style>
"""))

TARGET_NAME = "V CrA"
SESSION_DIR = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA"
)
cutout_dir = SESSION_DIR / "cutouts"
manifest_path = SESSION_DIR / "plate_manifest.csv"
accepted_list_path = SESSION_DIR / f"{TARGET_NAME.replace(' ', '_')}_dasch_accepted_exposures.csv"

pause_button = widgets.Button(description="⏸ Pause", button_style="warning",
                               layout=widgets.Layout(width="120px", height="36px"))
progress_html = widgets.HTML(value="<i>Setting up...</i>")
top_bar = widgets.HBox([pause_button, progress_html],
                        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"))
log_output = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="8px",
                                                    max_height="400px", overflow="auto"))
log_output.add_class("dark-log-output")
display(top_bar, log_output)

pause_event = threading.Event()
pause_button.on_click(lambda b: (pause_event.set(), setattr(pause_button, "description", "Pausing...")))

def format_eta(seconds):
    if seconds is None or seconds < 0:
        return "calculating..."
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def render_progress(n_done, total, start_time, status_word="Downloading", color="#4CAF50"):
    elapsed = time.monotonic() - start_time
    pct = (n_done / total * 100) if total else 0
    rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
    remaining_t = ((total - n_done) / rate) if rate > 0 else None
    filled = int(380 * pct / 100)
    progress_html.value = f"""
    <div style="font-family: monospace; font-size: 14px; color: white;">
      <div style="display:flex; align-items:center;">
        <div style="width:380px; height:18px; background:#333; border-radius:4px; overflow:hidden; margin-right:12px;">
          <div style="width:{filled}px; height:100%; background:{color};"></div>
        </div>
        <b>{pct:5.1f}%</b>
      </div>
      <div style="margin-top:4px;">
        {status_word} &nbsp; <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
        <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
        Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp; ETA <b>{format_eta(remaining_t)}</b>
      </div>
    </div>
    """

# NEW: proper flag-based guard, since raise SystemExit inside a Jupyter cell
# does NOT actually stop the rest of the cell from running (a real IPython
# quirk -- it gets caught/logged, then execution continues), which is what
# caused the NameError you saw right after the "mapping failed" message.
setup_ok = False
remaining = []
accepted_indices = []

with log_output:
    print(f"Opening session for {TARGET_NAME}...")
    sess = open_session(str(SESSION_DIR))
    sess.select_target(TARGET_NAME)
    sess.select_refcat("apass")

    exposures = sess.exposures()
    exposures_df = exposures.to_pandas()
    exposures_df.insert(0, "exposure_index", range(len(exposures_df)))
    print(f"Total exposures: {len(exposures_df)}")

    print("\nFetching lightcurve (local ID 0 = source closest to target)...")
    lc = sess.lightcurve(0)

    print("\nApplying DASCH's standard automatic rejections...")
    print("(DASCH's own warning: this is under development, treat with appropriate caution)")
    lc.apply_standard_rejections()

    lc_df = lc.to_pandas()
    n_total_lc = len(lc_df)
    n_detected = int(lc_df["magcal_magdep"].notna().sum())
    n_rejected = int((lc_df["reject"] != 0).sum())

    accepted_df = lc_df[(lc_df["reject"] == 0) & (lc_df["magcal_magdep"].notna())].copy()
    print(f"\nLightcurve rows total       : {n_total_lc}")
    print(f"Detections (before rejects) : {n_detected}")
    print(f"Flagged by standard rejects : {n_rejected}")
    print(f"ACCEPTED (not rejected, detected) : {len(accepted_df)}")

    accepted_df["exposure_index"] = accepted_df["exp_local_id"].astype(int)

    # FIX: exp_local_id == -1 (or any negative) is DASCH's sentinel for
    # "no exposure mapping available for this row" -- it is NOT evidence the
    # mapping scheme is broken. These rows are simply excluded, quietly.
    unmapped_mask = accepted_df["exposure_index"] < 0
    n_unmapped = int(unmapped_mask.sum())
    if n_unmapped:
        print(f"\n{n_unmapped} accepted rows have no exposure mapping "
              f"(exp_local_id == -1) -- excluding these from download, this is expected/normal.")
    accepted_df = accepted_df[~unmapped_mask].copy()

    print("\nVerifying exp_local_id -> exposure index mapping for the remaining "
          f"{len(accepted_df)} rows...")
    exposures_indexed = exposures_df.set_index("exposure_index")
    genuine_mismatches = []

    for _, row in accepted_df.iterrows():
        idx = row["exposure_index"]
        if idx not in exposures_indexed.index:
            genuine_mismatches.append((idx, "index not found in exposures()"))
            continue
        exp_row = exposures_indexed.loc[idx]
        checks = [
            (str(row.get("series", "")).strip(), str(exp_row.get("series", "")).strip()),
            (int(row.get("platenum", -1)), int(exp_row.get("platenum", -2))),
            (int(row.get("mosnum", -1)), int(exp_row.get("mosnum", -2))),
            (int(row.get("solnum", -1)), int(exp_row.get("solnum", -2))),
        ]
        if any(a != b for a, b in checks):
            genuine_mismatches.append((idx, checks))

    if genuine_mismatches:
        print(f"\n⚠️  WARNING: {len(genuine_mismatches)} GENUINE mismatches found -- "
              f"a real exposure index exists but its metadata doesn't match.")
        print("   STOPPING before download -- do not trust the results below.")
        print("   First few:")
        for m in genuine_mismatches[:5]:
            print(f"     {m}")
        setup_ok = False
    else:
        print(f"Verified: all {len(accepted_df)} mapped rows match their exposure "
              f"index on series/platenum/mosnum/solnum. Mapping confirmed correct.")

        accepted_indices = sorted(accepted_df["exposure_index"].unique().tolist())

        accepted_df[["exposure_index", "series", "platenum", "mosnum", "solnum",
                     "magcal_magdep", "magcal_magdep_rms", "aflags"]].sort_values(
            "exposure_index"
        ).to_csv(accepted_list_path, index=False)
        print(f"\nSaved accepted exposure list to: {accepted_list_path}")

        if manifest_path.exists():
            manifest_df = pd.read_csv(manifest_path)
        else:
            manifest_df = exposures_df.copy()
            manifest_df["status"] = "not_attempted"
            manifest_df["filename"] = ""
            manifest_df["error_message"] = ""
            manifest_df["last_updated"] = ""
        manifest_df = manifest_df.set_index("exposure_index", drop=False)

        existing_files = {f.name for f in cutout_dir.glob("*.fits")} if cutout_dir.exists() else set()
        if existing_files and "filename" in manifest_df.columns:
            on_disk_mask = manifest_df["filename"].apply(
                lambda f: Path(f).name in existing_files if isinstance(f, str) and f else False
            )
            reconciled = (manifest_df["status"] != "downloaded") & on_disk_mask
            manifest_df.loc[reconciled, "status"] = "downloaded"

        already_downloaded = set(
            manifest_df.index[manifest_df["status"] == "downloaded"].tolist()
        )
        remaining = [i for i in accepted_indices if i not in already_downloaded]
        already_have = len(accepted_indices) - len(remaining)

        print(f"\nOf {len(accepted_indices)} DASCH-accepted, mapped plates:")
        print(f"  Already downloaded : {already_have}")
        print(f"  Need to download   : {len(remaining)}")

        setup_ok = True

# Everything below now properly gated on setup_ok -- a real failure above
# means this whole block is skipped, no more stray NameErrors.
if setup_ok:
    total = max(len(remaining), 1)
    render_progress(0, total, time.monotonic(), status_word="Starting...")

    MAX_WORKERS = 6

    def fetch_one(i):
        try:
            result = sess.cutout(i)
            if result is None:
                return i, "unavailable", None, "no cutout available"
            return i, "downloaded", str(result), None
        except Exception as e:
            return i, "error", None, str(e)

    def save_manifest():
        try:
            manifest_df.reset_index(drop=True).sort_values("exposure_index").to_csv(
                manifest_path, index=False
            )
        except Exception as e:
            with log_output:
                print(f"Couldn't save manifest right now ({e}).")

    def run_downloads():
        counts = {"downloaded": 0, "unavailable": 0, "error": 0}
        n_done = 0
        start_time = time.monotonic()
        stopped_early = False

        ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
        futures = {ex.submit(fetch_one, i): i for i in remaining}
        pending = set(futures.keys())

        while pending:
            if pause_event.is_set():
                stopped_early = True
                with log_output:
                    print("\n⏸ Pause requested -- no new downloads will start.")
                ex.shutdown(wait=False, cancel_futures=True)
                break

            done, pending = wait(pending, timeout=0.2, return_when=FIRST_COMPLETED)
            for fut in done:
                i, status, filename, err_msg = fut.result()
                manifest_df.loc[i, "status"] = status
                manifest_df.loc[i, "filename"] = filename or ""
                manifest_df.loc[i, "error_message"] = err_msg or ""
                manifest_df.loc[i, "last_updated"] = datetime.now().isoformat(timespec="seconds")
                counts[status] += 1
                n_done += 1

            if done:
                render_progress(n_done, len(futures), start_time)
                save_manifest()
        else:
            ex.shutdown(wait=True)

        save_manifest()
        final_word = "Paused" if stopped_early else "Done"
        render_progress(n_done, len(futures), start_time, status_word=final_word,
                         color="#FFA726" if stopped_early else "#4CAF50")
        pause_button.description = final_word
        pause_button.disabled = True

        with log_output:
            print(f"\n{'Paused' if stopped_early else 'Finished'}.")
            print(f"This run -- downloaded: {counts['downloaded']}, "
                  f"unavailable: {counts['unavailable']}, error: {counts['error']}")
            n_on_disk_accepted = sum(
                1 for i in accepted_indices
                if i in manifest_df.index and manifest_df.loc[i, "status"] == "downloaded"
            )
            print(f"\nTotal DASCH-accepted plates now on disk: {n_on_disk_accepted} / {len(accepted_indices)}")
            print(f"Accepted list saved to: {accepted_list_path}")

    if remaining:
        threading.Thread(target=run_downloads, daemon=True).start()
    else:
        with log_output:
            print("\nAll DASCH-accepted plates are already downloaded. Nothing to do.")
        render_progress(0, 1, time.monotonic(), status_word="Done", color="#4CAF50")
        pause_button.description = "Done"
        pause_button.disabled = True
else:
    with log_output:
        print("\nSetup did not complete successfully -- no download was started. "
              "Scroll up to see what failed.")

Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

- Querying API ...
- Querying API ...
- Querying API ...


- Querying API ...
- Querying API ...
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\00002_b03834m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\00017_b05255m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\00001_b03757m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\00016_b05238m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\00003_b03896m2s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dap

    (281.88462342, -38.15897441)> appears to have failed: {'message': 'Endpoint request timed out'}


- Fetched 1399680 bytes in 11 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\03400_am17886m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\03402_b61380m1s0.fits`
- Querying API ...
- Querying API ...


...

- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05935_b73715m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05937_am26900m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05941_bi04441m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05949_rb15737m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05955_rb15753m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test

    (281.88462342, -38.15897441)> appears to have failed: {'message': 'Endpoint request timed out'}


- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05971_bi04478m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05968_am26956m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05972_rb15778m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05973_bi04480m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\05976_rb15783m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\tes

    (281.88462342, -38.15897441)> appears to have failed: {'message': 'Endpoint request timed out'}


- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\07063_dsy00268m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\07071_dsb00374m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\07072_dsy00374m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\07075_dsr00375m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts\07074_dsb00375m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Surve